# EINX Training — Free GPU

This notebook trains the EINX model on a **free Google Colab GPU**.

## What to do:
1. Click **Runtime** → **Change runtime type** → **GPU** (if not already set)
2. Click **Run All** (or press Ctrl+F9)
3. Wait ~10 minutes
4. See the test results at the bottom

That's it. No credit card, no setup, just click and wait.

In [ ]:
# Step 1: Check GPU is available
import torch
print(f'PyTorch: {torch.__version__}')
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU name: {torch.cuda.get_device_name(0)}')
    print(f'GPU memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
else:
    print('⚠ No GPU! Go to Runtime → Change runtime type → GPU, then Run All again.')

In [ ]:
# Step 2: Clone EINX repo and install
!git clone https://github.com/lewiseinstein15-Tech/einx-fooundation-model.git /content/einx
import sys
sys.path.insert(0, '/content/einx')

# Install dependencies
!pip install pydantic fastapi uvicorn pyyaml requests tqdm -q

# Verify EINX imports work
from einx.config import EINXModelConfig, TrainingConfig, RuntimeConfig
from einx.tokenizer.bpe import BPETokenizer
from einx.model.transformer import EINXTransformer
from einx.training.trainer import EINXTrainer
print('✓ EINX imported successfully')

In [ ]:
# Step 3: Generate the knowledge corpus (real facts, math, logic, science)
import sys
sys.path.insert(0, '/content/einx')

from scripts.generate_knowledge_v2 import generate_expanded_knowledge_corpus
import json
from pathlib import Path

records = generate_expanded_knowledge_corpus(30000, seed=42)
Path('/content/einx/data/raw').mkdir(parents=True, exist_ok=True)
with open('/content/einx/data/raw/knowledge_corpus_v2.jsonl', 'w') as f:
    for r in records:
        f.write(json.dumps(r) + '\n')
unique = set(r['text'] for r in records)
print(f'Generated {len(records):,} records ({len(unique):,} unique facts)')

In [ ]:
# Step 4: Train tokenizer
import sys
sys.path.insert(0, '/content/einx')
from einx.tokenizer.bpe import BPETokenizer
from einx.data.dataset import load_jsonl, write_jsonl, train_val_test_split
from einx.data.shards import ShardWriter
from einx.data.manifest import create_manifest

tok = BPETokenizer()
texts = [r['text'] for r in records]
tok.train(texts, vocab_size=1024, verbose=False)
tok.save('/content/einx/data/tokenized/einx-knowledge-bpe.json')
print(f'Tokenizer: vocab={tok.vocab_size()}, merges={len(tok.merges)}')

In [ ]:
# Step 5: Build dataset (WITHOUT dedup — keep duplicates for memorization)
import sys
sys.path.insert(0, '/content/einx')
from einx.tokenizer.bpe import BPETokenizer
from einx.data.dataset import load_jsonl, write_jsonl, train_val_test_split
from einx.data.shards import ShardWriter
from einx.data.manifest import create_manifest
from pathlib import Path

tok = BPETokenizer.load('/content/einx/data/tokenized/einx-knowledge-bpe.json')

# Load ALL records (with duplicates)
all_records = load_jsonl('/content/einx/data/raw/knowledge_corpus_v2.jsonl')
train_recs, val_recs, _ = train_val_test_split(
    all_records, val_ratio=0.05, test_ratio=0.0, seed=42,
)

output = Path('/content/einx/data/processed_knowledge')
output.mkdir(parents=True, exist_ok=True)
total_tokens = 0
for split, recs in [('train', train_recs), ('val', val_recs)]:
    split_dir = output / split
    split_dir.mkdir(parents=True, exist_ok=True)
    writer = ShardWriter(split_dir, shard_size=10000)
    for rec in recs:
        ids = tok.encode(rec['text'], add_eos=True)
        ids = ids[:129]
        writer.write({'input_ids': ids})
        total_tokens += len(ids)
    stats = writer.close()
    print(f'{split}: {stats["total_records"]:,} records, {stats["shard_count"]} shards')

print(f'Total tokens: {total_tokens:,}')

In [ ]:
# Step 6: Train the model on GPU!
# This is the main training — 2000 steps on GPU (should take ~5 minutes)
import sys, time, math, json
sys.path.insert(0, '/content/einx')
import torch
from einx.config import EINXModelConfig, TrainingConfig, RuntimeConfig
from einx.data.shards import ShardDataset
from einx.model.transformer import EINXTransformer
from einx.tokenizer.bpe import BPETokenizer
from einx.training.trainer import EINXTrainer
from einx.training.loss_logger import LossLogger

model_cfg = EINXModelConfig(
    name='einx-knowledge-gpu',
    vocab_size=1024,
    hidden_dim=256,         # bigger model — GPU can handle it
    n_layers=6,              # deeper
    n_heads=8,               # more heads
    head_dim=32,
    max_context_length=128,
    ffn_dim=1024,            # bigger FFN
    dropout=0.1,
    positional_encoding='rope',
    norm_type='rms',
    precision='bf16',        # GPU supports BF16!
    tie_word_embeddings=True,
)

train_cfg = TrainingConfig(
    run_name='knowledge-gpu',
    model_name='einx-knowledge-gpu',
    batch_size=64,           # bigger batch on GPU
    grad_accum_steps=2,
    learning_rate=0.001,
    weight_decay=0.1,
    max_grad_norm=1.0,
    warmup_steps=100,
    lr_schedule='cosine',
    min_lr_ratio=0.1,
    max_steps=2000,          # 2000 steps!
    save_every_steps=500,
    keep_last_n_checkpoints=3,
    eval_every_steps=200,
    eval_steps=50,
    log_every_steps=50,
    log_level='INFO',
    checkpoint_dir='/content/einx/checkpoints',
    device='cuda',            # GPU!
    precision='bf16',        # Mixed precision
    seed=42,
)

tokenizer = BPETokenizer.load('/content/einx/data/tokenized/einx-knowledge-bpe.json')
train_ds = ShardDataset('/content/einx/data/processed_knowledge/train', context_length=128)
val_ds = ShardDataset('/content/einx/data/processed_knowledge/val', context_length=128)

model = EINXTransformer(model_cfg)
print(f'Model: {model.n_params:,} params')
print(f'Architecture: {model_cfg.n_layers}L {model_cfg.hidden_dim}D {model_cfg.n_heads}H ctx={model_cfg.max_context_length}')
print(f'Dataset: train={len(train_ds):,} samples, val={len(val_ds):,} samples')
print(f'Training: 2000 steps on GPU with BF16 mixed precision')
print()

runtime_cfg = RuntimeConfig(device='cuda', precision='bf16', compile=False)
trainer = EINXTrainer(
    model, train_cfg, train_ds, val_ds,
    tokenizer=tokenizer,
    runtime_config=runtime_cfg,
)

start = time.time()
result = trainer.train()
elapsed = time.time() - start

entries = LossLogger.load(result.get('loss_log_path', ''))
train_e = [e for e in entries if 'val_loss' not in e]
val_e = [e for e in entries if 'val_loss' in e]

print()
print('=' * 60)
print('TRAINING COMPLETE')
print('=' * 60)
print(f'Time: {elapsed:.1f}s ({elapsed/60:.1f} min)')
print(f'Steps: {result["final_step"]}')
if train_e:
    print(f'Initial loss: {train_e[0]["loss"]:.4f}')
    print(f'Final loss:   {train_e[-1]["loss"]:.4f}')
    reduction = (1 - train_e[-1]['loss']/train_e[0]['loss']) * 100
    print(f'Reduction:    {reduction:.1f}%')
if val_e:
    ppl = math.exp(val_e[-1]['val_loss'])
    print(f'Val loss:     {val_e[-1]["val_loss"]:.4f}')
    print(f'Perplexity:   {ppl:.2f}')
    print(f'Tokens seen:  {train_e[-1]["tokens_seen"]:,}')
perf = result.get('performance', {})
if perf:
    print(f'Tokens/sec:   {perf.get("avg_tokens_per_second", 0):.0f}')
    print(f'Peak memory:  {perf.get("peak_memory_mb", 0):.0f} MB')

In [ ]:
# Step 7: TEST THE MODEL WITH REAL QUESTIONS
import sys, json
sys.path.insert(0, '/content/einx')
import torch
from einx.tokenizer.bpe import BPETokenizer
from einx.model.transformer import EINXTransformer
from einx.training.checkpoint_manager import CheckpointManager

tok = BPETokenizer.load('/content/einx/data/tokenized/einx-knowledge-bpe.json')
mgr = CheckpointManager('/content/einx/checkpoints', run_name='knowledge-gpu')
best = mgr.find_best() or mgr.find_latest()
print(f'Loading checkpoint: {best}')
model = EINXTransformer.load(best, map_location='cuda')
model.eval()

tests = [
    # BASIC MATH
    ('question: what is 2 + 2? answer:', '4'),
    ('question: what is 5 + 3? answer:', '8'),
    ('question: what is 7 + 8? answer:', '15'),
    ('question: what is 9 x 9? answer:', '81'),
    ('question: what is 6 x 7? answer:', '42'),
    ('question: what is 10 - 3? answer:', '7'),
    ('question: what is half of 10? answer:', '5'),
    ('question: what is 100 / 10? answer:', '10'),
    
    # WORLD KNOWLEDGE
    ('question: what is the capital of france? answer:', 'paris'),
    ('question: what is the capital of japan? answer:', 'tokyo'),
    ('question: what is the capital of england? answer:', 'london'),
    ('question: what is the capital of germany? answer:', 'berlin'),
    ('question: what is the capital of italy? answer:', 'rome'),
    ('question: what is the capital of china? answer:', 'beijing'),
    ('question: what is the capital of russia? answer:', 'moscow'),
    ('question: what is the capital of india? answer:', 'delhi'),
    ('question: what is the capital of brazil? answer:', 'brasilia'),
    ('question: what is the capital of egypt? answer:', 'cairo'),
    ('question: what planet do we live on? answer:', 'earth'),
    ('question: what is the closest star to earth? answer:', 'sun'),
    ('question: how many days are in a week? answer:', '7'),
    ('question: how many months are in a year? answer:', '12'),
    ('question: what color is the sky? answer:', 'blue'),
    ('question: what color is grass? answer:', 'green'),
    ('question: what color is blood? answer:', 'red'),
    ('question: what color is snow? answer:', 'white'),
    ('question: what color is a banana? answer:', 'yellow'),
    ('question: what animal says meow? answer:', 'cat'),
    ('question: what animal says woof? answer:', 'dog'),
    ('question: what animal says moo? answer:', 'cow'),
    ('question: what animal says quack? answer:', 'duck'),
    
    # SCIENCE
    ('question: how many legs does a spider have? answer:', '8'),
    ('question: how many legs does an insect have? answer:', '6'),
    ('question: how many legs does a dog have? answer:', '4'),
    ('question: how many planets are in the solar system? answer:', '8'),
    ('question: what is the largest planet? answer:', 'jupiter'),
    ('question: what is the smallest planet? answer:', 'mercury'),
    ('question: what is the boiling point of water? answer:', '100'),
    ('question: what is the freezing point of water? answer:', '0'),
    ('question: how many bones does a human have? answer:', '206'),
    ('question: how many chambers does the heart have? answer:', '4'),
    ('question: what is the largest ocean? answer:', 'pacific'),
    ('question: what is the tallest mountain? answer:', 'everest'),
    
    # OPPOSITES
    ('question: what is the opposite of hot? answer:', 'cold'),
    ('question: what is the opposite of up? answer:', 'down'),
    ('question: what is the opposite of big? answer:', 'small'),
    ('question: what is the opposite of fast? answer:', 'slow'),
    ('question: what is the opposite of light? answer:', 'dark'),
    ('question: what is the opposite of good? answer:', 'bad'),
    ('question: what is the opposite of old? answer:', 'new'),
    ('question: what is the opposite of day? answer:', 'night'),
    ('question: what is the opposite of wet? answer:', 'dry'),
    
    # CALENDAR
    ('question: what comes after monday? answer:', 'tuesday'),
    ('question: what comes after friday? answer:', 'saturday'),
    ('question: what comes after sunday? answer:', 'monday'),
    ('question: what comes after december? answer:', 'january'),
    
    # LOGIC
    ('if today is monday, tomorrow is', 'tuesday'),
    ('if today is tuesday, tomorrow is', 'wednesday'),
    ('if today is wednesday, tomorrow is', 'thursday'),
    ('if today is friday, tomorrow is', 'saturday'),
    ('if 2x = 10, then x =', '5'),
    ('if 3x = 15, then x =', '5'),
    ('if x + 5 = 12, then x =', '7'),
    ('if x + 3 = 10, then x =', '7'),
    ('if a + b = 10 and a = 3, then b =', '7'),
]

correct = 0
total = len(tests)

print()
print('=' * 60)
print('EINX KNOWLEDGE TEST — REAL QUESTIONS')
print('=' * 60)
print()

for prompt, expected in tests:
    ids = tok.encode(prompt, add_bos=False)
    input_ids = torch.tensor([ids], dtype=torch.long, device='cuda')
    generated = []
    for _ in range(10):
        with torch.no_grad():
            logits, _ = model(input_ids)
        next_id = torch.argmax(logits[0, -1, :]).item()
        if next_id == tok.special.eos_id:
            break
        generated.append(next_id)
        input_ids = torch.cat([input_ids, torch.tensor([[next_id]], dtype=torch.long, device='cuda')], dim=1)
    answer = tok.decode(generated).strip().lower()
    is_correct = expected.lower() in answer
    if is_correct:
        correct += 1
    status = '✓' if is_correct else '✗'
    print(f'{status} Q: {prompt}')
    print(f'  Expected: {expected}  |  Got: {answer!r}')
    print()

print('=' * 60)
print(f'SCORE: {correct}/{total} ({correct/total*100:.0f}%)')
print('=' * 60)

# Save results
with open('/content/einx/experiments/colab_results.json', 'w') as f:
    json.dump({'score': f'{correct}/{total}', 'percentage': correct/total*100}, f, indent=2)

print()
print('Done! Copy the results above and send them back.')